# Domain Blocking Statistical Analysis

Exploratory analysis of `wsj_domain_status` + `wsj_crawl_results` to find data-driven blocking criteria.

**Problem**: Current auto-blocking misses structurally uncrawlable domains:
- Wilson block only counts infrastructure failures (`http error`, `timeout`)
- Quality block threshold too low (`avg_llm < 3.0`)
- No criterion for "zero success after N attempts" with content/quality failures

**Goal**: Find thresholds that block waste without harming legitimate sources.

In [ ]:
# Cell 1: Imports + Supabase connection
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from supabase import create_client

load_dotenv(Path('..') / '.env.local')

url = os.getenv('NEXT_PUBLIC_SUPABASE_URL') or os.getenv('SUPABASE_URL')
key = os.getenv('SUPABASE_SERVICE_ROLE_KEY')
assert url and key, 'Missing Supabase credentials'
supabase = create_client(url, key)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_colwidth', 40)
print('Connected to Supabase')

In [ ]:
# Cell 2: Data Load — paginate both tables

def paginate_table(table, select_cols, page_size=1000):
    all_rows = []
    offset = 0
    while True:
        batch = (supabase.table(table).select(select_cols)
                 .range(offset, offset + page_size - 1)
                 .execute().data or [])
        all_rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return all_rows

# Domain status — all domains
domain_rows = paginate_table(
    'wsj_domain_status',
    'domain, status, fail_count, fail_counts, success_count, '
    'success_rate, wilson_score, avg_llm_score, avg_crawl_length, '
    'avg_embedding_score, search_hit_count, block_reason, last_failure, last_success'
)
df_dom = pd.DataFrame(domain_rows)
print(f'wsj_domain_status: {len(df_dom)} domains')

# Crawl results — all rows
crawl_rows = paginate_table(
    'wsj_crawl_results',
    'id, resolved_domain, crawl_status, crawl_error, crawl_length, embedding_score, created_at'
)
df_crawl = pd.DataFrame(crawl_rows)
print(f'wsj_crawl_results: {len(df_crawl)} rows')

# LLM analysis — relevance scores
llm_rows = paginate_table(
    'wsj_llm_analysis',
    'crawl_result_id, relevance_score, content_quality'
)
df_llm = pd.DataFrame(llm_rows).rename(columns={'crawl_result_id': 'id'})
print(f'wsj_llm_analysis: {len(df_llm)} rows')

# Merge crawl + llm
df_crawl = df_crawl.merge(df_llm, on='id', how='left')
print(f'\nCrawl+LLM merged: {len(df_crawl)} rows')

## Section 2: Overview Stats

In [ ]:
# Cell 3: Domain status overview

print('=== Domain Count by Status ===')
print(df_dom['status'].value_counts())

# Numeric cols
num_cols = ['fail_count', 'success_count', 'success_rate', 'wilson_score',
            'avg_llm_score', 'avg_crawl_length', 'search_hit_count']
for col in num_cols:
    df_dom[col] = pd.to_numeric(df_dom[col], errors='coerce')

df_dom['total_attempts'] = df_dom['success_count'] + df_dom['fail_count']

print('\n=== Active Domain Stats ===')
active = df_dom[df_dom['status'] == 'active']
print(active[num_cols + ['total_attempts']].describe().round(2))

In [ ]:
# Cell 4: Distribution plots

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

plot_cols = ['total_attempts', 'success_rate', 'wilson_score',
             'avg_llm_score', 'avg_crawl_length', 'search_hit_count']
titles = ['Total Attempts', 'Success Rate', 'Wilson Score',
          'Avg LLM Score', 'Avg Crawl Length', 'Search Hit Count']

for ax, col, title in zip(axes.flat, plot_cols, titles):
    data = active[col].dropna()
    if len(data) > 0:
        ax.hist(data, bins=30, edgecolor='white', alpha=0.8)
        ax.set_title(title)
        ax.axvline(data.median(), color='red', linestyle='--', alpha=0.7, label=f'median={data.median():.2f}')
        ax.legend(fontsize=8)

fig.suptitle('Active Domain Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## Section 3: Domain Segmentation

In [ ]:
# Cell 5: Segment active domains with >= 5 attempts

df_seg = active[active['total_attempts'] >= 5].copy()
print(f'Active domains with >= 5 attempts: {len(df_seg)}')

def segment(row):
    if row['success_count'] == 0 and pd.isna(row['avg_llm_score']):
        return 'Never reached LLM'
    if row['success_count'] == 0:
        return 'Zero success'
    if row['avg_llm_score'] >= 6:
        return 'Good'
    return 'Marginal'

df_seg['segment'] = df_seg.apply(segment, axis=1)
print('\n=== Segment Counts ===')
print(df_seg['segment'].value_counts())

print('\n=== Segment Stats ===')
display(df_seg.groupby('segment')[['total_attempts', 'success_rate', 'avg_llm_score', 'wilson_score']]
        .agg(['count', 'mean', 'median']).round(2))

In [ ]:
# Cell 6: Segment distribution boxplots

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, title in zip(axes, ['success_rate', 'avg_llm_score', 'wilson_score'],
                           ['Success Rate', 'Avg LLM Score', 'Wilson Score']):
    order = ['Good', 'Marginal', 'Zero success', 'Never reached LLM']
    existing = [s for s in order if s in df_seg['segment'].values]
    sns.boxplot(data=df_seg, x='segment', y=col, order=existing, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## Section 4: Failure Pattern Analysis

In [ ]:
# Cell 7: Parse fail_counts JSONB

def parse_fail_counts(val):
    if isinstance(val, dict):
        return val
    if isinstance(val, str):
        try:
            return json.loads(val)
        except (json.JSONDecodeError, TypeError):
            return {}
    return {}

df_seg = df_seg.copy()
df_seg['fail_dict'] = df_seg['fail_counts'].apply(parse_fail_counts)

# Explode into failure type columns
all_fail_types = set()
for d in df_seg['fail_dict']:
    all_fail_types.update(d.keys())

for ft in sorted(all_fail_types):
    df_seg[f'fail_{ft}'] = df_seg['fail_dict'].apply(lambda d: d.get(ft, 0))

fail_cols = [c for c in df_seg.columns if c.startswith('fail_') and c != 'fail_count' and c != 'fail_counts' and c != 'fail_dict']

print('=== Failure Types by Segment ===')
display(df_seg.groupby('segment')[fail_cols].sum().T)

In [ ]:
# Cell 8: Content-too-short ratio analysis

cts_col = [c for c in fail_cols if 'content' in c.lower() and 'short' in c.lower()]
if cts_col:
    cts_col = cts_col[0]
    df_seg['cts_ratio'] = df_seg[cts_col] / df_seg['total_attempts']
    df_seg['cts_ratio'] = df_seg['cts_ratio'].fillna(0)

    print(f'=== content_too_short Ratio by Segment (col: {cts_col}) ===')
    display(df_seg.groupby('segment')['cts_ratio'].describe().round(2))

    # Domains with >= 80% content_too_short
    high_cts = df_seg[df_seg['cts_ratio'] >= 0.8].sort_values('total_attempts', ascending=False)
    print(f'\n=== Domains with content_too_short >= 80% ({len(high_cts)}) ===')
    display(high_cts[['domain', 'segment', 'total_attempts', 'success_count', cts_col, 'cts_ratio']].head(20))
else:
    print('No content_too_short failure type found in fail_counts')
    print(f'Available: {fail_cols}')

## Section 5: Threshold Exploration

In [ ]:
# Cell 9: Define "should block" ground truth label
# Conservative: segment in {Zero success, Never reached LLM} = should block
# Edge: Marginal domains with very low avg_llm may also deserve blocking

df_seg['should_block'] = df_seg['segment'].isin(['Zero success', 'Never reached LLM'])
print(f'Should block: {df_seg["should_block"].sum()} / {len(df_seg)}')
print(f'Should keep: {(~df_seg["should_block"]).sum()}')

In [ ]:
# Cell 10: Threshold sweep — zero success + N attempts

results = []
for n in range(3, 20):
    predicted_block = (df_seg['success_count'] == 0) & (df_seg['total_attempts'] >= n)
    tp = (predicted_block & df_seg['should_block']).sum()
    fp = (predicted_block & ~df_seg['should_block']).sum()
    fn = (~predicted_block & df_seg['should_block']).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    results.append({'criterion': f'zero_success_N>={n}', 'N': n,
                    'predicted_block': predicted_block.sum(),
                    'tp': tp, 'fp': fp, 'fn': fn,
                    'precision': precision, 'recall': recall})

df_zero = pd.DataFrame(results)
display(df_zero.style.format({'precision': '{:.2%}', 'recall': '{:.2%}'}))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_zero['N'], df_zero['precision'], 'o-', label='Precision')
ax.plot(df_zero['N'], df_zero['recall'], 's-', label='Recall')
ax.set_xlabel('Min attempts (N)')
ax.set_ylabel('Score')
ax.set_title('Zero Success + N Attempts: Precision/Recall')
ax.legend()
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Threshold sweep — avg_llm < X with min samples

results_llm = []
for x in np.arange(3.0, 6.5, 0.5):
    for min_samples in [3, 5, 8]:
        has_llm = df_seg['avg_llm_score'].notna() & (df_seg['total_attempts'] >= min_samples)
        predicted_block = has_llm & (df_seg['avg_llm_score'] < x)
        tp = (predicted_block & df_seg['should_block']).sum()
        fp = (predicted_block & ~df_seg['should_block']).sum()
        fn = (~predicted_block & df_seg['should_block']).sum()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        results_llm.append({
            'avg_llm_thresh': x, 'min_samples': min_samples,
            'predicted_block': predicted_block.sum(),
            'tp': tp, 'fp': fp, 'fn': fn,
            'precision': precision, 'recall': recall,
        })

df_llm_thresh = pd.DataFrame(results_llm)
display(df_llm_thresh.style.format({'precision': '{:.2%}', 'recall': '{:.2%}', 'avg_llm_thresh': '{:.1f}'}))

In [ ]:
# Cell 12: Threshold sweep — content_too_short ratio >= Y

if 'cts_ratio' in df_seg.columns:
    results_cts = []
    for y in np.arange(0.5, 1.05, 0.1):
        for min_attempts in [5, 8, 10]:
            predicted_block = (df_seg['cts_ratio'] >= y) & (df_seg['total_attempts'] >= min_attempts)
            tp = (predicted_block & df_seg['should_block']).sum()
            fp = (predicted_block & ~df_seg['should_block']).sum()
            fn = (~predicted_block & df_seg['should_block']).sum()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            results_cts.append({
                'cts_ratio_thresh': round(y, 1), 'min_attempts': min_attempts,
                'predicted_block': predicted_block.sum(),
                'tp': tp, 'fp': fp, 'fn': fn,
                'precision': precision, 'recall': recall,
            })

    df_cts_thresh = pd.DataFrame(results_cts)
    display(df_cts_thresh.style.format({'precision': '{:.2%}', 'recall': '{:.2%}'}))
else:
    print('No cts_ratio column — skip')

In [ ]:
# Cell 13: Combined criterion — overall Wilson (all failures) < Z
# Recompute Wilson using ALL failure types, not just blockable ones

from scipy.stats import norm

def wilson_lower(successes, total, z=1.96):
    """Wilson score 95% CI lower bound."""
    if total == 0:
        return 0.0
    p = successes / total
    denom = 1 + z**2 / total
    center = p + z**2 / (2 * total)
    spread = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2))
    return (center - spread) / denom

df_seg['overall_wilson'] = df_seg.apply(
    lambda r: wilson_lower(r['success_count'], r['total_attempts']), axis=1
)

results_wilson = []
for z_thresh in np.arange(0.0, 0.25, 0.02):
    for min_attempts in [5, 8, 10]:
        predicted_block = (df_seg['overall_wilson'] < z_thresh) & (df_seg['total_attempts'] >= min_attempts)
        tp = (predicted_block & df_seg['should_block']).sum()
        fp = (predicted_block & ~df_seg['should_block']).sum()
        fn = (~predicted_block & df_seg['should_block']).sum()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        results_wilson.append({
            'wilson_thresh': round(z_thresh, 2), 'min_attempts': min_attempts,
            'predicted_block': predicted_block.sum(),
            'tp': tp, 'fp': fp, 'fn': fn,
            'precision': precision, 'recall': recall,
        })

df_wilson_thresh = pd.DataFrame(results_wilson)
display(df_wilson_thresh.style.format({'precision': '{:.2%}', 'recall': '{:.2%}'}))

## Section 6: Edge Case Review

In [ ]:
# Cell 14: Identify domains newly blocked under candidate thresholds

# Define candidate combined criteria (tune after reviewing Cells 10-13)
criteria = {
    'A: zero_success >= 5 attempts': (
        (df_seg['success_count'] == 0) & (df_seg['total_attempts'] >= 5)
    ),
    'B: zero_success >= 8 attempts': (
        (df_seg['success_count'] == 0) & (df_seg['total_attempts'] >= 8)
    ),
    'C: avg_llm < 4.0 (min 5 samples)': (
        df_seg['avg_llm_score'].notna() &
        (df_seg['avg_llm_score'] < 4.0) &
        (df_seg['total_attempts'] >= 5)
    ),
    'D: overall_wilson < 0.05 (min 5)': (
        (df_seg['overall_wilson'] < 0.05) & (df_seg['total_attempts'] >= 5)
    ),
}

review_cols = ['domain', 'segment', 'total_attempts', 'success_count',
               'success_rate', 'avg_llm_score', 'overall_wilson']

for name, mask in criteria.items():
    blocked = df_seg[mask].sort_values('total_attempts', ascending=False)
    print(f'\n=== {name}: {len(blocked)} domains ===')
    display(blocked[review_cols].head(15))

In [ ]:
# Cell 15: Flag recognizable domains — manual review

# Known major domains to watch for false positives
major_domains = [
    'theguardian.com', 'techcrunch.com', 'reuters.com', 'bloomberg.com',
    'cnbc.com', 'bbc.com', 'bbc.co.uk', 'nytimes.com', 'washingtonpost.com',
    'ft.com', 'wsj.com', 'apnews.com', 'finance.yahoo.com', 'yahoo.com',
    'cnn.com', 'forbes.com', 'marketwatch.com', 'barrons.com',
]

print('=== Major Domain Status ===')
for name, mask in criteria.items():
    flagged = df_seg[mask & df_seg['domain'].isin(major_domains)]
    if len(flagged) > 0:
        print(f'\n** {name} would block major domains: **')
        display(flagged[review_cols])
    else:
        print(f'{name}: no major domains affected')

In [ ]:
# Cell 16: apps.apple.com-type analysis — what catches them?

# Find app store / structurally uncrawlable patterns
app_patterns = ['apps.apple.com', 'play.google.com', 'store.steampowered.com',
                'music.apple.com', 'podcasts.apple.com']

app_domains = df_seg[df_seg['domain'].str.contains('|'.join(app_patterns), na=False)]
if len(app_domains) == 0:
    # Broader: check all zero-success domains with high attempt counts
    app_domains = df_seg[(df_seg['success_count'] == 0) & (df_seg['total_attempts'] >= 5)]
    print(f'No app store domains found. Showing all zero-success domains (>= 5 attempts): {len(app_domains)}')
else:
    print(f'App store domains: {len(app_domains)}')

display(app_domains[review_cols + (['cts_ratio'] if 'cts_ratio' in df_seg.columns else [])].head(20))

# Which criteria catch them?
print('\n=== Which criterion catches each? ===')
for _, row in app_domains.iterrows():
    caught_by = []
    for name, mask in criteria.items():
        if mask.loc[row.name]:
            caught_by.append(name.split(':')[0])
    print(f"  {row['domain']:40s} caught by: {', '.join(caught_by) or 'NONE'}")

## Section 7: Recommendation

In [ ]:
# Cell 17: Summary table of proposed thresholds

# Compile best results from sweeps above
summary_data = []

for name, mask in criteria.items():
    tp = (mask & df_seg['should_block']).sum()
    fp = (mask & ~df_seg['should_block']).sum()
    fn = (~mask & df_seg['should_block']).sum()
    total_blocked = mask.sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    # False positive domains (for manual review)
    fp_domains = df_seg[mask & ~df_seg['should_block']]['domain'].tolist()

    summary_data.append({
        'Criterion': name,
        'Blocked': total_blocked,
        'True Pos': tp,
        'False Pos': fp,
        'False Neg': fn,
        'Precision': precision,
        'Recall': recall,
        'FP Domains': ', '.join(fp_domains[:5]) + ('...' if len(fp_domains) > 5 else ''),
    })

df_summary = pd.DataFrame(summary_data)
display(df_summary.style.format({'Precision': '{:.1%}', 'Recall': '{:.1%}'}).set_caption(
    'Proposed Blocking Criteria — Impact Summary'))

print('\n=== Recommendation ===')
print('Review the precision/recall trade-offs above.')
print('High precision (few false positives) is preferred — we want broad crawling.')
print('Combine criteria with OR for higher recall, AND for higher precision.')